In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [2]:
with open(r"C:\Users\rajme\Documents\PYTORCH_NOTEBOOKS\PYTORCH_NOTEBOOKS\Data\shakespeare.txt") as f:
    text = f.read()

In [6]:
all_charecters = set(text)

In [8]:
decoder = dict(enumerate(all_charecters))

In [12]:
encoder = {char:index for index, char in decoder.items()}

In [16]:
encoded_text = np.array([encoder[char] for char in text])

In [29]:
encoded_text.flatten()

array([31,  2,  2, ..., 71, 59, 50])

In [33]:
def onehotencoder(encoded_text, nounique):

    one_hot = np.zeros((encoded_text.size,nounique))
    one_hot = one_hot.astype(np.float32)

    one_hot[np.arange(one_hot.shape[0]), encoded_text.flatten()] = 1.0

    one_hot = one_hot.reshape((*encoded_text.shape,nounique))

    return one_hot

In [109]:
def generate_batches(encoded_text, samp_per_batches = 10, seq_length = 50):

    #encoded text = [1,22,3,4,5......] something like this
    #samp per bacth is basically the number of rows in one batch which also is the batch size
    #seq length is the number of columns in one batch
    
    char_per_batch = samp_per_batches * seq_length #this calculates the total number of values in the bacth matrix so i*j
    
    no_of_batches = int(len(encoded_text)/char_per_batch) #this calculates the number of bacthes by dividing the total number of
                                                          # items with the number of samples per batch hence we get number of bactches
                                                          # like unitary method
    
    encoded_text = encoded_text[:char_per_batch*no_of_batches] #this makes sure we have the correct length by removing everything 
                                                               #beyond the charperbatch*number of batches so that its rounded
    
    encoded_text = encoded_text.reshape((samp_per_batches, -1)) #this reshapes to (samp per batch, length/sampleperbactch)

    #from here we will actually start creatng (sampperbatch, seq length) previously it was the original seq stream
    for n in range(0, encoded_text.shape[1], seq_length): #goes from 0, to total columns, with a step of sequence length
        
        # Grab feature characters
        x = encoded_text[:, n:n+seq_length] # first time x gets from 0 to 0+seq length
        
        # y is the target shifted over by 1
        y = np.zeros_like(x) #gives the same shape as x because thats how our data needs to be
       
        #
        try:
            y[:, :-1] = x[:, 1:] #takes everything from the 1st column of the x matrix and puts it from the 0th column of the y matrix
                                #so if x has of t -> t+seq length, y has from t+1 -> seq length+1
            y[:, -1]  = encoded_text[:, n+seq_length] #the seq length plus onne is coming from the main batch which has all the data 

            #seq is basically the columns 
            
        # FOR POTENTIAL INDEXING ERROR AT THE END    
        except:
            y[:, :-1] = x[:, 1:]
            y[:, -1] = encoded_text[:, 0]
            
        yield x, y

In [111]:
class CharGenModel(nn.Module):
    
    def __init__(self, all_chars, num_hidden=256, num_layers=4,drop_prob=0.5,use_gpu=False):
        
        
        # SET UP ATTRIBUTES
        super().__init__()
        self.drop_prob = drop_prob
        self.num_layers = num_layers
        self.num_hidden = num_hidden
        self.use_gpu = use_gpu
        
        #CHARACTER SET, ENCODER, and DECODER
        self.all_chars = all_chars
        self.decoder = dict(enumerate(all_chars))
        self.encoder = {char: ind for ind,char in decoder.items()        
        
        self.lstm = nn.LSTM(len(self.all_chars), num_hidden, num_layers, dropout=drop_prob, batch_first=True)
       
        self.dropout = nn.Dropout(drop_prob)
        
        self.fc_linear = nn.Linear(num_hidden, len(self.all_chars))
      
    
    def forward(self, x, hidden):
                  
        
        lstm_output, hidden = self.lstm(x, hidden)
        
        drop_output = self.dropout(lstm_output)
        
        drop_output = drop_output.contiguous().view(-1, self.num_hidden)   
        
        final_out = self.fc_linear(drop_output)      
        
        return final_out, hidden
    
    
    def hidden_state(self, batch_size):
        '''
        Used as separate method to account for both GPU and CPU users.
        '''
        
        if self.use_gpu:
            
            hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden).cuda(),
                     torch.zeros(self.num_layers,batch_size,self.num_hidden).cuda())
        else:
            hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden),
                     torch.zeros(self.num_layers,batch_size,self.num_hidden))
        
        return hidden
        

In [114]:
model = CharGenModel(
    all_chars=all_charecters,
    num_hidden=512,
    num_layers=3,
    drop_prob=0.5,
    use_gpu=False,
)

In [115]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
criterion = nn.CrossEntropyLoss()

In [116]:
train_percent = 0.9
train_idx = int(len(encoded_text)*train_percent)
train_data = encoded_text[:train_idx]
val_data = encoded_text[train_idx:]

In [122]:
epochs = 10
batch_size = 100
seq_length = 100

tracker = 0

num_char = max(encoded_text)+1

In [123]:
model.training

True

In [ ]:
model.train()

for i in range(epochs):

    hidden = model.hidden_state(batch_size) #making sure our hidden state is converted to zeros before each epoch

    for x, y in generate_batches(train_data, batch_size, seq_length): #x and y shape would be (batch size, seq length)
        
        tracker += 1
        x = torch.from_numpy(onehotencoder(x, num_char)) #this makes it (batch size, seq length, no of charecters)
                                                        #no of charecters is basically the length
                                                        #of the one hot encoded array [0,0,1,......,0] baically like a lit up
                                                        #pixel for what encoded number it represent, and 
        y = torch.from_numpy(y)

        hidden = tuple(state.detach() from state in hidden) #since we are passing through the model layers in batches
                                                            #we need to detach the computational graph or else memory fail

        model.zero_grad()

        lstm_output, hidden = model.forward(x, hidden)

        loss = criterion(lstm_output, y.view(batch_size*seq_length))
        loss.backward()
        
        nn.utils.clip_grad_norm(model.parameters(), max_norm=5)

        